# Conclusion canonique — NextWatch

Cette synthèse lit uniquement `results/final_evaluation.json`, produit par `scripts/Models/evaluate_final.py` selon le protocole commun de `scripts/evaluation_protocol.py`.

## Protocole attendu

921 cas communs ; split 65 % apprentissage, 15 % validation et 20 % test ; `random_state=42` ; 1 à 3 films aimés tirés de façon déterministe ; référence `train+validation` et test tenu à part.


## Résultats finaux

Le tableau ci-dessous est construit directement depuis `final_evaluation.json`. En plus de P@5 et P@10, le **Hit Rate@5** indique le pourcentage de cas ayant au moins un film aimé parmi les cinq recommandations.

## Conclusion et limites

Sur les 921 cas communs, le classement reste **UserCF > Cluster > baseline** pour P@5, P@10 et Hit Rate@5. UserCF atteint un Hit Rate@5 de **37,6 %**, contre **22,7 %** pour Cluster et **15,5 %** pour la baseline.

Les deux modèles ont été réglés avec 300 essais Optuna sur la validation. Pour Cluster, le meilleur essai est le numéro 240 avec P@5 validation = **0,0467** ; l’artefact final réentraîné sur apprentissage+validation atteint P@5 = **0,0534** sur le test intact.

Limites principales : split aléatoire non temporel, utilisateurs du test déjà présents dans la référence, un seul split, et absence de mesure de couverture, diversité, nouveauté ou latence.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "results" / "final_evaluation.json").is_file()
)
RESULT_PATH = ROOT / "results" / "final_evaluation.json"
result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))

expected_protocol = {
    "random_state": 42,
    "train_frac": 0.65,
    "validation_frac": 0.15,
    "test_frac": 0.2,
    "max_seeds": 3,
    "liked_definition": "rating > user_mean(reference)",
    "seed_selection": "deterministic_random_1_to_3",
    "reference_split": "train+validation",
    "heldout_split": "test",
    "same_cases_for_all_methods": True,
}
expected_scores = {
    "baseline_overlap_mean": {
        "P@5": 0.034310532030401734,
        "P@10": 0.037242128121606946,
        "HitRate@5": 0.15526601520086863,
    },
    "user_cluster_groups": {
        "P@5": 0.053420195439739415,
        "P@10": 0.04809989142236699,
        "HitRate@5": 0.2269272529858849,
    },
    "user_user_cf": {
        "P@5": 0.09315960912052117,
        "P@10": 0.08534201954397394,
        "HitRate@5": 0.3756786102062975,
    },
}
assert result["n_cases"] == 921
assert all(result["protocol"].get(key) == value for key, value in expected_protocol.items())
assert result["scores"] == expected_scores

labels = {
    "baseline_overlap_mean": "baseline",
    "user_cluster_groups": "Cluster",
    "user_user_cf": "UserCF",
}
scores = pd.DataFrame(
    {
        "méthode": labels[key],
        "P@5": values["P@5"],
        "P@10": values["P@10"],
        "Hit Rate@5": values["HitRate@5"],
    }
    for key, values in result["scores"].items()
)
print(f"{result['n_cases']} cas ; protocole commun vérifié.")
display(
    scores.style.format(
        {"P@5": "{:.4f}", "P@10": "{:.4f}", "Hit Rate@5": "{:.1%}"}
    )
)
print("Classement final : UserCF > Cluster > baseline.")

921 cas ; protocole commun vérifié.


,méthode,P@5,P@10,Hit Rate@5
0,baseline,0.0343,0.0372,15.5%
1,Cluster,0.0534,0.0481,22.7%
2,UserCF,0.0932,0.0853,37.6%


Classement final : UserCF > Cluster > baseline.


Cellule excédentaire neutralisée.


In [2]:
# Cellule excédentaire neutralisée.

Cellule excédentaire neutralisée.


In [3]:
# Cellule excédentaire neutralisée.

Cellule excédentaire neutralisée.


Cellule excédentaire neutralisée.

Cellule excédentaire neutralisée.

Cellule excédentaire neutralisée.